# ONNX Autopsy Kaggle Experiment Harness

Use this notebook to benchmark and tune approaches on Kaggle only.

Workflow:
1. Run all cells to compute exact NED on random and stress splits.
2. Compare weighted proxy score: `0.3 * random + 0.7 * stress`.
3. Log results to `/kaggle/working/experiments_log.csv`.

Notes:
- This notebook is self-contained and does **not** require uploading `solution.py`.
- The retrieval solution logic is implemented directly in Cell 6 for fast Kaggle iteration.
- After finding improvements here, we port the winning config back into workspace `solution.py`.

In [7]:
import importlib.util
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Ready. numpy:', np.__version__, 'pandas:', pd.__version__)

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda device count:', torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            print(f'GPU {i}:', torch.cuda.get_device_name(i))
        x = torch.tensor([1.0], device='cuda')
        print('gpu sanity op:', float((x * 2).item()))
except Exception as e:
    print('torch/cuda check skipped:', repr(e))

print('\nNote: the retrieval stage in this harness is CPU-bound (sparse sklearn ops).')
print('GPU utilization is expected to stay low until neural training cells are added.')

Ready. numpy: 2.0.2 pandas: 2.3.3
torch: 2.10.0+cu128
cuda available: True
cuda device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
gpu sanity op: 2.0

Note: the retrieval stage in this harness is CPU-bound (sparse sklearn ops).
GPU utilization is expected to stay low until neural training cells are added.


In [2]:
def resolve_data_dir() -> Path:
    direct_candidates = [
        Path('/kaggle/input'),
        Path('./dataset/public'),
        Path('../dataset/public'),
        Path('./dataset'),
        Path('../dataset'),
    ]

    found = []
    for base in direct_candidates:
        if base.is_dir() and (base / 'train.csv').exists() and (base / 'test.csv').exists():
            found.append(base)

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for train_file in kaggle_input.rglob('train.csv'):
            parent = train_file.parent
            if (parent / 'test.csv').exists():
                found.append(parent)

    unique = []
    seen = set()
    for p in found:
        rp = p.resolve()
        if str(rp) in seen:
            continue
        seen.add(str(rp))
        unique.append(rp)

    if not unique:
        raise FileNotFoundError('No directory with train.csv and test.csv found.')

    unique.sort(key=lambda p: (0 if (p / 'sample_submission.csv').exists() else 1, len(str(p))))
    return unique[0]

DATA_DIR = resolve_data_dir()
train_df = pd.read_csv(DATA_DIR / 'train.csv')

print('DATA_DIR:', DATA_DIR)
print('train shape:', train_df.shape)

DATA_DIR: /kaggle/input/datasets/soufianelasfar/onnx-dataset
train shape: (8000, 3)


In [3]:
def parse_target(raw: str):
    try:
        val = json.loads(raw)
    except Exception:
        return []
    if not isinstance(val, list):
        return []
    return [x for x in val if isinstance(x, str)]

def levenshtein(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        ai = a[i - 1]
        for j in range(1, m + 1):
            tmp = dp[j]
            cost = 0 if ai == b[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = tmp
    return dp[m]

def ned(pred, true):
    if not pred and not true:
        return 1.0
    d = levenshtein(pred, true)
    return 1.0 - d / max(len(pred), len(true))

train_df['target_list'] = train_df['target_sequence'].map(parse_target)
train_df['hex_len'] = train_df['onnx_hex'].str.len()
train_df['seq_len'] = train_df['target_list'].str.len()

print('hex_len median:', float(train_df['hex_len'].median()))
print('seq_len median:', float(train_df['seq_len'].median()))

hex_len median: 4282.0
seq_len median: 11.0


In [4]:
def build_splits(df: pd.DataFrame, val_size: int = 1200, seed: int = 42):
    n = len(df)
    val_size = min(val_size, max(1, n // 3))

    rng = np.random.default_rng(seed)
    all_idx = np.arange(n)
    rng.shuffle(all_idx)

    random_valid = np.sort(all_idx[:val_size]).tolist()
    random_set = set(random_valid)
    random_train = [i for i in range(n) if i not in random_set]

    stress_valid = df.nlargest(val_size, 'hex_len').index.to_list()
    stress_set = set(stress_valid)
    stress_train = [i for i in range(n) if i not in stress_set]

    return {
        'random': (random_train, random_valid),
        'stress': (stress_train, stress_valid),
    }

SPLITS = build_splits(train_df, val_size=1200, seed=SEED)
print('random split sizes:', len(SPLITS['random'][0]), len(SPLITS['random'][1]))
print('stress split sizes:', len(SPLITS['stress'][0]), len(SPLITS['stress'][1]))

random split sizes: 6800 1200
stress split sizes: 6800 1200


In [8]:
import heapq
import math
from collections import Counter, defaultdict
from functools import lru_cache

from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import linear_kernel

# Profiles control reranking depth/compute only. Retrieval cache is built once.
PROFILE_PRESETS = {
    'fast': {
        'top_k': 24,
        'batch_size': 256,
        'metric_neighbors': 14,
        'max_candidates': 32,
    },
    'balanced': {
        'top_k': 32,
        'batch_size': 192,
        'metric_neighbors': 20,
        'max_candidates': 48,
    },
    'quality': {
        'top_k': 44,
        'batch_size': 128,
        'metric_neighbors': 30,
        'max_candidates': 72,
    },
}


def byte_even_view(hex_str: str) -> str:
    if len(hex_str) <= 4:
        return hex_str
    return ''.join(hex_str[i:i+2] for i in range(0, len(hex_str), 4))


def byte_edge_view(hex_str: str, prefix_bytes: int = 2048, suffix_bytes: int = 2048) -> str:
    total_bytes = len(hex_str) // 2
    if total_bytes <= prefix_bytes + suffix_bytes:
        return hex_str
    return hex_str[:prefix_bytes * 2] + hex_str[-suffix_bytes * 2:]


class SequencePrior:
    def __init__(self, sequences):
        self.vocab = set()
        self.start_counts = Counter()
        self.end_counts = Counter()
        self.bigram_counts = defaultdict(Counter)

        for seq in sequences:
            if not seq:
                continue
            self.vocab.update(seq)
            self.start_counts[seq[0]] += 1
            self.end_counts[seq[-1]] += 1
            for a, b in zip(seq, seq[1:]):
                self.bigram_counts[a][b] += 1

        self.vocab_size = max(1, len(self.vocab))
        self.start_total = max(1, sum(self.start_counts.values()))
        self.end_total = max(1, sum(self.end_counts.values()))

    def score(self, seq):
        if not seq:
            return -8.0

        alpha = 0.5
        s = 0.0
        s += math.log((self.start_counts[seq[0]] + alpha) / (self.start_total + alpha * self.vocab_size))
        for a, b in zip(seq, seq[1:]):
            row = self.bigram_counts.get(a)
            row_total = sum(row.values()) if row else 0
            count = row[b] if row else 0
            s += math.log((count + alpha) / (row_total + alpha * self.vocab_size))
        s += math.log((self.end_counts[seq[-1]] + alpha) / (self.end_total + alpha * self.vocab_size))
        return s / max(1, len(seq))


class LengthPrior:
    def __init__(self, train_hex_lens, train_seq_lens):
        bucket_values = defaultdict(list)
        for h, s in zip(train_hex_lens, train_seq_lens):
            bucket_values[self.bucket(h)].append(s)

        self.bucket_means = {}
        global_mean = sum(train_seq_lens) / max(1, len(train_seq_lens))
        for b in range(20):
            vals = bucket_values.get(b)
            self.bucket_means[b] = (sum(vals) / len(vals)) if vals else global_mean

    @staticmethod
    def bucket(hex_len):
        return min(19, int(math.log2(max(2, hex_len))))

    def expected_length(self, hex_len):
        return self.bucket_means[self.bucket(hex_len)]


@lru_cache(maxsize=400000)
def sequence_lev_distance(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        ai = a[i - 1]
        for j in range(1, m + 1):
            tmp = dp[j]
            cost = 0 if ai == b[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = tmp
    return dp[m]


def sequence_ned(a, b):
    if not a and not b:
        return 1.0
    d = sequence_lev_distance(a, b)
    return 1.0 - d / max(len(a), len(b))


def choose_sequence(neighbors, train_sequences, prior_scores, default_length, frequent_sequences, max_metric_neighbors, max_candidates):
    if not neighbors:
        return frequent_sequences[0] if frequent_sequences else ('Linear', 'ReLU', 'Linear')

    candidate_scores = defaultdict(float)
    weight_sum = 0.0
    length_sum = 0.0
    nn_items = []

    for rank, (idx, sim) in enumerate(neighbors):
        if idx < 0 or idx >= len(train_sequences):
            continue
        seq = train_sequences[idx]
        if not seq:
            continue
        sim = max(0.0, float(sim))
        rank_decay = 1.0 / (1.0 + 0.17 * rank)
        w = (sim + 1e-6) * rank_decay
        candidate_scores[seq] += w
        weight_sum += w
        length_sum += w * len(seq)
        nn_items.append((seq, w))

    if not candidate_scores:
        return frequent_sequences[0] if frequent_sequences else ('Linear', 'ReLU', 'Linear')

    for seq in frequent_sequences[:12]:
        candidate_scores[seq] += 0.0

    if max_candidates > 0 and len(candidate_scores) > max_candidates:
        top_items = heapq.nlargest(max_candidates, candidate_scores.items(), key=lambda x: x[1])
        candidate_scores = defaultdict(float, top_items)

    expected_len = length_sum / weight_sum if weight_sum > 0 else default_length
    if not math.isfinite(expected_len) or expected_len <= 0:
        expected_len = default_length

    if weight_sum > 0:
        nn_items = [(seq, w / weight_sum) for seq, w in nn_items]
    else:
        uniform = 1.0 / max(1, len(nn_items))
        nn_items = [(seq, uniform) for seq, _ in nn_items]

    if max_metric_neighbors > 0:
        nn_items = nn_items[:max_metric_neighbors]

    if nn_items:
        max_len = int(round(expected_len))
        max_len = max(1, min(64, max_len))
        consensus = []
        for pos in range(max_len):
            tok_w = defaultdict(float)
            for seq, w in nn_items:
                if pos < len(seq):
                    tok_w[seq[pos]] += w
            if not tok_w:
                break
            consensus.append(max(tok_w.items(), key=lambda x: x[1])[0])
        if consensus:
            candidate_scores[tuple(consensus)] += 0.0

    best_seq = None
    best_score = -1e18
    for seq, base_score in candidate_scores.items():
        lp = prior_scores.get(seq, -8.0)
        len_penalty = -abs(len(seq) - expected_len) / max(1.0, expected_len)

        exp_ned = 0.0
        support = 0.0
        for nseq, w in nn_items:
            exp_ned += w * sequence_ned(seq, nseq)
            if nseq == seq:
                support += w

        score = (
            1.00 * exp_ned
            + 0.06 * support
            + 0.03 * base_score
            + 0.02 * len_penalty
            + 0.003 * lp
        )
        if score > best_score:
            best_score = score
            best_seq = seq

    if best_seq is None:
        return frequent_sequences[0] if frequent_sequences else ('Linear', 'ReLU', 'Linear')
    return best_seq


class SklearnRetriever:
    def __init__(self):
        self.vec_full = HashingVectorizer(
            analyzer='char',
            ngram_range=(3, 6),
            n_features=1 << 19,
            lowercase=False,
            alternate_sign=False,
            binary=False,
            norm=None,
            dtype=np.float32,
        )
        self.vec_even = HashingVectorizer(
            analyzer='char',
            ngram_range=(3, 5),
            n_features=1 << 18,
            lowercase=False,
            alternate_sign=False,
            binary=False,
            norm=None,
            dtype=np.float32,
        )
        self.vec_edge = HashingVectorizer(
            analyzer='char',
            ngram_range=(3, 6),
            n_features=1 << 18,
            lowercase=False,
            alternate_sign=False,
            binary=False,
            norm=None,
            dtype=np.float32,
        )

        self.tfidf_full = TfidfTransformer(norm='l2', sublinear_tf=True)
        self.tfidf_even = TfidfTransformer(norm='l2', sublinear_tf=True)
        self.tfidf_edge = TfidfTransformer(norm='l2', sublinear_tf=True)

        self.X_train_full = None
        self.X_train_even = None
        self.X_train_edge = None

    def fit(self, train_hex):
        train_full = list(train_hex)
        train_even = [byte_even_view(x) for x in train_hex]
        train_edge = [byte_edge_view(x) for x in train_hex]

        Xf = self.vec_full.transform(train_full)
        Xe = self.vec_even.transform(train_even)
        Xg = self.vec_edge.transform(train_edge)

        self.X_train_full = self.tfidf_full.fit_transform(Xf)
        self.X_train_even = self.tfidf_even.fit_transform(Xe)
        self.X_train_edge = self.tfidf_edge.fit_transform(Xg)

    def query(self, test_hex, top_k=40, batch_size=128):
        test_full = list(test_hex)
        test_even = [byte_even_view(x) for x in test_hex]
        test_edge = [byte_edge_view(x) for x in test_hex]

        Xt_full = self.tfidf_full.transform(self.vec_full.transform(test_full))
        Xt_even = self.tfidf_even.transform(self.vec_even.transform(test_even))
        Xt_edge = self.tfidf_edge.transform(self.vec_edge.transform(test_edge))

        n_test = Xt_full.shape[0]
        n_train = self.X_train_full.shape[0]
        k = min(top_k, n_train)
        out = []
        w_full, w_even, w_edge = 0.62, 0.23, 0.15

        for start in range(0, n_test, batch_size):
            end = min(n_test, start + batch_size)
            sim_full = linear_kernel(Xt_full[start:end], self.X_train_full)
            sim_even = linear_kernel(Xt_even[start:end], self.X_train_even)
            sim_edge = linear_kernel(Xt_edge[start:end], self.X_train_edge)
            sim = w_full * sim_full + w_even * sim_even + w_edge * sim_edge

            for row in sim:
                if k <= 0:
                    out.append([])
                    continue
                idx = np.argpartition(row, -k)[-k:]
                idx = idx[np.argsort(row[idx])[::-1]]
                out.append([(int(i), float(row[i])) for i in idx])
        return out


SPLIT_CACHE = {}


def build_split_cache(split_name: str, val_size: int = 1200, seed: int = 42, top_k_max: int = 44):
    key = (split_name, val_size, seed, top_k_max)
    if key in SPLIT_CACHE:
        return SPLIT_CACHE[key]

    splits = build_splits(train_df, val_size=val_size, seed=seed)
    train_idx, valid_idx = splits[split_name]

    train_part = train_df.iloc[train_idx]
    valid_part = train_df.iloc[valid_idx]

    train_hex = train_part['onnx_hex'].tolist()
    valid_hex = valid_part['onnx_hex'].tolist()
    train_targets = [tuple(x) for x in train_part['target_list'].tolist()]
    valid_true = valid_part['target_list'].tolist()
    valid_hex_lens = valid_part['hex_len'].tolist()

    train_hex_lens = [len(x) for x in train_hex]
    train_seq_lens = [len(x) for x in train_targets]

    seq_counter = Counter(train_targets)
    frequent_sequences = [seq for seq, _ in seq_counter.most_common(120) if seq]
    if not frequent_sequences:
        frequent_sequences = [('Linear', 'ReLU', 'Linear')]

    prior = SequencePrior(train_targets)
    prior_scores = {seq: prior.score(seq) for seq in seq_counter.keys()}
    len_prior = LengthPrior(train_hex_lens, train_seq_lens)
    default_lens = [len_prior.expected_length(h) for h in valid_hex_lens]

    retriever = SklearnRetriever()
    t0 = time.time()
    retriever.fit(train_hex)
    neighbors_all = retriever.query(valid_hex, top_k=top_k_max, batch_size=128)
    build_sec = time.time() - t0

    cache = {
        'train_targets': train_targets,
        'valid_true': valid_true,
        'neighbors_all': neighbors_all,
        'prior_scores': prior_scores,
        'frequent_sequences': frequent_sequences,
        'default_lens': default_lens,
        'build_sec': build_sec,
    }
    SPLIT_CACHE[key] = cache
    return cache


def evaluate_from_cache(cache, cfg):
    top_k = cfg['top_k']
    metric_neighbors = cfg['metric_neighbors']
    max_candidates = cfg['max_candidates']

    t0 = time.time()
    preds = []
    for neighbors, default_len in zip(cache['neighbors_all'], cache['default_lens']):
        pred = choose_sequence(
            neighbors=neighbors[:top_k],
            train_sequences=cache['train_targets'],
            prior_scores=cache['prior_scores'],
            default_length=default_len,
            frequent_sequences=cache['frequent_sequences'],
            max_metric_neighbors=metric_neighbors,
            max_candidates=max_candidates,
        )
        preds.append(pred)

    scores = [ned(list(p), t) for p, t in zip(preds, cache['valid_true'])]
    rerank_sec = time.time() - t0
    return float(np.mean(scores)), rerank_sec


def evaluate_solution_profile(profile: str, val_size: int = 1200, seed: int = 42):
    if profile not in PROFILE_PRESETS:
        raise ValueError(f'Unknown profile: {profile}')

    cfg = PROFILE_PRESETS[profile]
    top_k_max = max(v['top_k'] for v in PROFILE_PRESETS.values())

    random_cache = build_split_cache('random', val_size=val_size, seed=seed, top_k_max=top_k_max)
    stress_cache = build_split_cache('stress', val_size=val_size, seed=seed, top_k_max=top_k_max)

    random_ned, random_rerank = evaluate_from_cache(random_cache, cfg)
    stress_ned, stress_rerank = evaluate_from_cache(stress_cache, cfg)
    weighted = 0.3 * random_ned + 0.7 * stress_ned

    return {
        'profile': profile,
        'random_ned': random_ned,
        'stress_ned': stress_ned,
        'weighted_proxy': weighted,
        'random_time_sec': random_cache['build_sec'] + random_rerank,
        'stress_time_sec': stress_cache['build_sec'] + stress_rerank,
        'rerank_random_sec': random_rerank,
        'rerank_stress_sec': stress_rerank,
        'val_size': val_size,
        'seed': seed,
    }

print('Inline retrieval solution loaded. Profiles:', list(PROFILE_PRESETS.keys()))
print('Optimization enabled: split caches are built once and reused across profiles.')

Inline retrieval solution loaded. Profiles: ['fast', 'balanced', 'quality']
Optimization enabled: split caches are built once and reused across profiles.


In [12]:
# Optional extra profiles for a wider sweep without rebuilding retrieval cache.
PROFILE_PRESETS.update({
    'quality_plus': {
        'top_k': 52,
        'batch_size': 128,
        'metric_neighbors': 36,
        'max_candidates': 96,
    },
    'quality_trim': {
        'top_k': 36,
        'batch_size': 128,
        'metric_neighbors': 24,
        'max_candidates': 64,
    },
})

profiles = ['fast', 'balanced', 'quality', 'quality_plus', 'quality_trim']
results = []

# Build caches once (the expensive step).
max_top_k = max(PROFILE_PRESETS[p]['top_k'] for p in profiles)
print('Prebuilding split caches once...')
t_cache_start = time.time()
_ = build_split_cache('random', val_size=1200, seed=SEED, top_k_max=max_top_k)
_ = build_split_cache('stress', val_size=1200, seed=SEED, top_k_max=max_top_k)
print(f'Cache build done in {time.time() - t_cache_start:.2f}s')

for profile in profiles:
    print(f'Running profile: {profile}')
    try:
        result = evaluate_solution_profile(profile=profile, val_size=1200, seed=SEED)
    except Exception as e:
        result = {
            'profile': profile,
            'error': repr(e),
            'random_ned': np.nan,
            'stress_ned': np.nan,
            'weighted_proxy': np.nan,
            'random_time_sec': np.nan,
            'stress_time_sec': np.nan,
            'rerank_random_sec': np.nan,
            'rerank_stress_sec': np.nan,
            'val_size': 1200,
            'seed': SEED,
        }
    results.append(result)

results_df = pd.DataFrame(results).sort_values('weighted_proxy', ascending=False, na_position='last')
results_df

Prebuilding split caches once...
Cache build done in 0.00s
Running profile: fast
Running profile: balanced
Running profile: quality
Running profile: quality_plus
Running profile: quality_trim


,profile,random_ned,stress_ned,weighted_proxy,random_time_sec,stress_time_sec,rerank_random_sec,rerank_stress_sec,val_size,seed
4,quality_trim,0.867370,0.552663,0.647075,371.767542,363.847318,11.324242,1.388442,1200,42
2,quality,0.867004,0.552455,0.646820,370.971631,368.580079,10.528331,6.121203,1200,42
1,balanced,0.867282,0.552152,0.646691,361.328199,363.355844,0.884900,0.896969,1200,42
0,fast,0.867429,0.551395,0.646205,361.211279,363.169613,0.767980,0.710737,1200,42
3,quality_plus,0.864902,0.552158,0.645981,384.235577,370.244563,23.792277,7.785687,1200,42


In [10]:
LOG_PATH = Path('/kaggle/working/experiments_log.csv')
log_df = results_df.copy()
log_df['timestamp_utc'] = pd.Timestamp.utcnow().isoformat()

if LOG_PATH.exists():
    prev = pd.read_csv(LOG_PATH)
    merged = pd.concat([prev, log_df], ignore_index=True)
else:
    merged = log_df

merged.to_csv(LOG_PATH, index=False)
print('Saved log to:', LOG_PATH)
merged.tail(10)

Saved log to: /kaggle/working/experiments_log.csv


,profile,random_ned,stress_ned,weighted_proxy,random_time_sec,stress_time_sec,rerank_random_sec,rerank_stress_sec,val_size,seed,timestamp_utc
4,quality_trim,0.867370,0.552663,0.647075,371.906333,363.811002,11.463034,1.352127,1200,42,2026-04-18T05:16:34.726248+00:00
2,quality,0.867004,0.552455,0.646820,372.948997,368.739406,12.505697,6.280530,1200,42,2026-04-18T05:16:34.726248+00:00
1,balanced,0.867282,0.552152,0.646691,366.838960,364.628662,6.395660,2.169787,1200,42,2026-04-18T05:16:34.726248+00:00
0,fast,0.867429,0.551395,0.646205,370.397952,365.244586,9.954653,2.785711,1200,42,2026-04-18T05:16:34.726248+00:00
3,quality_plus,0.864902,0.552158,0.645981,384.721798,370.381958,24.278498,7.923082,1200,42,2026-04-18T05:16:34.726248+00:00


In [11]:
best_row = results_df.iloc[0].to_dict()
best_profile = best_row.get('profile')

print('Best profile:', best_profile)
print('Weighted proxy:', best_row.get('weighted_proxy'))
print('Random NED:', best_row.get('random_ned'))
print('Stress NED:', best_row.get('stress_ned'))

print('\nCopy these env vars for final local generation run:')
print(f"SHIPD_PROFILE={best_profile}")
print('SHIPD_REQUIRE_SKLEARN=1')

Best profile: quality_trim
Weighted proxy: 0.6470751628568794
Random NED: 0.8673697385590435
Stress NED: 0.5526632018416662

Copy these env vars for final local generation run:
SHIPD_PROFILE=quality_trim
SHIPD_REQUIRE_SKLEARN=1
